# Analisi dati per effetto Hall

A grandi linee:
1. caratterizzare l'**uniformità del campo magnetico** nel traferro
2. caratterizzare l'andamento del **campo magnetico** nel traferro al **variare della corrente** in ingresso
3. caratterizzare **tensione di Hall al variare della corrente** $V_H(i)$ (ferromagnete spento + 5x2 valori del campo magnetico)
4. caratterizzare **tensione di Hall al variare del campo magnetico** $V_H(B)$ (ferromagnete spento + 3x2 valori di corrente)
5. valutare **mobilità** dei portatori di carica nel materiale

+ cose facolative (?)

### utils

In [ ]:
from utils import meanCalc, fitPlotter, testZ
import ROOT

# TODO: move stuff to utils
def MeanError(s1,s2):
    "used to calculate mean of two arrays of np.array([mean,error],[],...) type"
    if (np.shape(s1) != np.shape(s2)):
        raise("TypeError: shape of imput arrays must be equal")
    else:
        rel_error = np.sqrt((s1[:,1]/s1[:,0])**2 + (s2[:,1]/s2[:,0])**2)
        stima = (s1 + s2)/len(s1)
        stima[:,1] = rel_error * stima[:,0]
        return stima
        

def Zscore(s1, s2):
    "used to compare array of np.array([mean,error] measures. returns z score "
    if (np.shape(s1) != np.shape(s2)):
        raise("TypeError: shape of imput arrays must be equal")
    else:
        errors = np.sqrt(s1[:,1]**2 + s2[:,1] **2)
        z = (s2[:,0]-s1[:,0])/errors
        for i in range(len(s1)):
            print(f"z value of param {i} :", round(z[i],3))
        print("\n")

        return z

## Uniformità campo magnetico (TOTHINKABOUT)

Facendo misure del campo magnetico con una sonda di Hall vogliamo studiare l'uniformità di $B$ all'interno del traferro del ferromagnete.

In generale, la semidifferenza tra il valore massimo e il valore minimo del campo magnetico (all'interno della regione di omogeneità) sarà il limite minimo dell'errore su tutte le misure di campo magnetico.

In [ ]:
# arrays of distances
posx = np.arange(5)
posy = np.arange(5)

# matrix of B values 
Bu = np.array([[1,1,1,1,1],
               [1,3,3,3,1],
               [1,3,3,3,1],
               [1,3,3,3,1],
               [1,1,1,1,1]])

Can2d_u = ROOT.TCanvas("c2d", "Uniformità campo magnetico")
H2d_u = ROOT.TH2F("h2d_u", "Uniformità campo magnetico; x [mm]; y [mm]; B [T]" ,5,0,5, 5,0,5)

for i in range(5):
    for j in range(5):
        H2d_u.SetBinContent(i+1,j+1,float(Bu[i][j]))

H2d_u.Draw("LEGO4")
Can2d_u.Draw()

Dopo lo studio di uniformità del campo magnetico, si stabilisce in quale zona effettuare le misure di campo magnetico. In tale zona, si prende la semidispersione di B come stima della risoluzione del campo magnetico.

In [ ]:
B_max = 1.6
B_min = 1.4
resB = (B_max - B_min)/2
print(resB)

## Campo magnetico prodotto al variare della corrente

Tralasciando la prima curva di salita (da $0$ a $i_\text{max}$) misuriamo due salite e due discese complete (da $i_\text{max}$ a $-i_\text{max}$, e viceversa) prendendo 10 dati per ogni curva. Sostanzialmente, vogliamo verificare la regione di linearità in cui (dopo) vogliamo svolgere il resto dell'esperienza.

Escluse le zone di saturazione vogliamo un fare un fit lineare su ogni curva: per ognuna delle due coppie (due curve di salita, e due curve di discesa) stimiamo coi valori medi di $m$ e $q$ il vero coefficiente angolare e la vera quota (*l'errore sulla quota lo stimiamo con la semidifferenza tra i due valori*).
Solo dopo mediamo i risultati ottenuti per le curve di salita e quelle di discesa (*errore sulla quota sempre dato dalla semidifferenza*).

In [ ]:
# first negative run
B1 = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
i1 = [2.2, 3.8, 6.1, 7.9, 10.2, 12.1]

# first positive run
B2 = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
i2 = [2.2, 3.8, 6.1, 7.9, 10.2, 12.1]

# second negative run
B3 = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
i3 = [2.2, 3.8, 6.1, 7.9, 10.2, 12.1]

# second positive run
B4 = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
i4 = [2.2, 3.8, 6.1, 7.9, 10.2, 12.1]

magCurrPlotter = fitPlotter("MagneticFieldVsCurrent")
param1 = magCurrPlotter.addGraph(B1, i1,title="run1")
param2 = magCurrPlotter.addGraph(B2, i2,title="run2")
param3 = magCurrPlotter.addGraph(B3, i3,title="run3")
param4 = magCurrPlotter.addGraph(B4, i4,title="run4")
magCurrPlotter.drawCanvas()
magCurrPlotter.saveCanvas()

NB: per i parametri ritornati dai fit, sto utilizzando il formato `np.array([[mean][error],...])` per scrivere i valori, così poi da avere le funzioni Zscore e MeanErrors per calcolare direttamente valori attessi ed errori

In [ ]:
# verifico che i parametri in salita e discesa siano compatibili, poi medio tutto

Zscore(param1, param3)
param_up = MeanError(param1, param3)

Zscore(param2, param4)
param_down = MeanError(param2, param4)

Zscore(param_up, param_down)
param = MeanError(param_up,param_down)

param

## Tensione di Hall al variare di $i$

Vogliamo valutare l'andamento della tensione di Hall (sui lati del materiale semiconduttore) al variare della corrente iniettata al suo interno, in modo da stimare il parametro $R_H$.

Quindi fissato il valore del campo magnetico (una volta a zero per eliminare il fondo, e poi a 5 valori diversi in entrambi i versi) valutiamo l'andamento $V_H(i)$ (*andando tra **-8mA e +8mA***) sapendo che in generale:
$$V_H = \frac{R_H}{t} \cdot i_p \cdot B$$
allora facciamo dei fit lineari su ogni set di dati e diamo una stima del parametro $R_H$ per entrambi i versi del campo magnetico, poi mediamo tra i due set.

In [ ]:
# correction for misalignment of transverse contacts
B0    = 0
errB0 = 0.1

VH0    = [1,2,3,4,5,6,7,8,9,10]
errVH0 = [0.1]*10
Ip0    = [1,2,3,4,5,6,7,8,9,10]
errIp0 = [0.2]*10

hallCurrPlotter = fitPlotter("HallTensionVsCurrent")
param0 = hallCurrPlotter.addGraph(Ip0, VH0, errIp0, errVH0, "Ohmic behaviour")
hallCurrPlotter.drawCanvas()
hallCurrPlotter.saveCanvas()

Stima_VH0 = param0[1]
Stima_VH0

In [ ]:
BHallpos    = [1,2,3,4,5]
errBHallpos = [0.1]*5
BHallneg    = [-1,-2,-3,-4,-5]
errBHallneg = [0.1]*5

IHall1    = [2,3,6,8,12,16,20,24,28,32] 
errIHall1 = [0.1]*10
VHall1    = [1,2,3,4,5,6,7,8,9,10]
errVHall1 = [0.3]*10

# we have to remove the contribution for B=0
# (our measurements need to have the same currents, or else we need to interpolate between the points...)
def removeBackground(data, back, err_data, err_back):
    """ removes background data `back` from `data` (propagating errors as sums of squares) """
    newdata     = []
    err_newdata = []

    for i in range(0,len(data)):
        newdata.append(data[i] - back[i])
        err_newdata.append((err_data[i]**2 + err_back[i]**2)**0.5)

    return newdata, err_newdata

cleanIHall1, errcleanIHall1 = removeBackground(IHall1,Ip0,errIHall1,errIp0)

newHallCurrPlotter = fitPlotter("HallTensionVsCurrent")
param1 = newHallCurrPlotter.addGraph(IHall1, VHall1, errIHall1, errVHall1, "w/ background")
param2 = newHallCurrPlotter.addGraph(cleanIHall1, VHall1, errcleanIHall1, errVHall1, "w/out background")
newHallCurrPlotter.drawCanvas()
newHallCurrPlotter.saveCanvas("fit1.png")

# posso verificare che i parametri siano compatibili, e poi estrarre il valore di resistenza di hall
zscore(param1,param2)

param = MeanError(param1,param2)

## Tensione di Hall al variare di $B$ (TODO)

Ripetiamo sostanzialmente le misure al punto precedente, ma invertendo i ruoli di variabile dipendente e indipendente. Adesso fissiamo $i_p$, scegliendo $3 \times 2$ valori, (dopo aver tracciato un'altra curva di caduta di potenziale ohmica a $B=0$) e studiamo il variare di $V_H$ con $B$.

**probabilmente ha senso riprendere i valori della curva ohmica (disallineamento) fissando i valori della corrente in base a questa nuova scansione: altrimenti dobbiamo interpolare lungo la curva di un fit**

In [ ]:
# TODO: but you can copy (almost everything) from above

## Mobilità dei portatori

Misurando la caratteristica $I(V)$ del materiale seminconduttore (facendo variare la corrente $I$ tra -8mA e +8mA) che abbiamo usato nell'esperienza possiamo dare una stima della mobilità dei portatori di carica al suo interno. In particolare, dalla pendenza di $I(V)$ abbiamo la resistenza $R$, e quindi anche la resistività:
$$\rho =\frac{t\cdot w}{L} R$$
dove $t$ è lo spessore, $w$ la larghezza e $L$ la lunghezza.
Infine, dalla resistività si ha direttamente:
$$\mu = \frac{R_H}{\rho} (=R_H \sigma)$$
dove per $R_H$ possiamo considerare le stime date prima.

In [ ]:
I    = [-8,-6,-4,-2,0,2,4,6,8]
errI = [0.1]*9
V    = [0,1,2,3,4,5,6,7,8]
errV = [0.1]*9

muPlotter = fitPlotter("IVcharacteristic")
param = muPlotter.addGraph(I, V, errI, errV, title="I(V)")
muPlotter.drawCanvas()
muPlotter.saveCanvas()

R, errR = param[1][0],param[1][1]

In [ ]:
t = 1
w = 1
L = 1

rho = t * w * R / L

R_H = 1

mu = R_H / rho

## facoltativo: Misura del coefficiente di Hall a temperatura variabile

La misura della temperatura è effettuata mediante una misura di resistenza su un elemento
Pt100 (R$\simeq$ 100 $\ohm$). la tabella di calibrazione è su moodle.

Impostare la corrente iniettata nel campione a 8 mA e immergere il campione stesso nel
traferro, impostando un campo magnetico a 200 mT.

In [ ]:
RPt = [100,120,140,160, 180] # fore non necessario
T = np.arange(25,140,5)
VH = np.arange(23) + np.random.rand(23)*3
VH

In [ ]:
np.shape(T)